# %% [markdown]
# cleaning follows a routine:
# 1. Look
# 2. Find the problems
# 3. Fix them one at a time
# 4. Never change the original
# 5. Write steps as code

In [ ]:

import pandas as pd
import sys

# Check Pandas version
print("Pandas version:", pd.__version__)

# Check Python version
print("Python version:", sys.version)

# Check Python environment
print("Python location:", sys.executable)

# Test whether Pandas works
test = pd.DataFrame({
    "Name": ["Anna", "John"],
    "Age": [25, 30]
})

print("Pandas test successful!")
print(test)


In [ ]:

import pandas as pd

orders_raw = pd.read_csv('data/orders.csv', sep=',')
orders = orders_raw.copy()

print(orders.shape)
print(orders.columns.tolist())
orders.info()

In [ ]:
orders.info()

In [ ]:
orders["category"].value_counts()

In [ ]:

import pandas as pd
from io import StringIO

# Read the original CSV as text
with open("data/orders.csv", "r", encoding="utf-8-sig") as file:
    lines = file.read().splitlines()

# Remove the extra outer quotation marks
fixed_lines = [
    line[1:-1].replace('""', '"')
    if line.startswith('"') and line.endswith('"')
    else line
    for line in lines
]

# Load the corrected CSV into Pandas
orders_raw = pd.read_csv(StringIO("\n".join(fixed_lines)))
orders = orders_raw.copy()

print(orders.shape)
orders.head(8)


In [ ]:
 
# %%
import pandas as pd
orders_raw = pd.read_csv('data/orders.csv')
 
# %%
orders = orders_raw.copy()
orders_raw.shape
 
# %%    
orders.head(8)
 
# %%
orders.info()
 
# %% [markdown]
# Text I: spaces and capital letters
 
# %%
orders['category'].nunique()
 
# %%
orders['category'].value_counts()
 
# %%
print((orders['category'] == 'Clothing').sum())
 
# %%
print(orders['category'].nunique())
print(orders['category'].str.strip().nunique())
print(orders['category'].str.strip().str.lower().nunique())
 
# %%
orders['category'].str.strip().str.lower().value_counts()
 
# %%
orders['category'] = (
    orders['category']
    .str.strip()
    .str.lower()
    .replace('home and garden', 'home & garden')
    .str.title()
)
orders['category'].value_counts()
 
# %%
orders['product'].nunique()
orders['product'].value_counts()
 
# %%
orders['express'].value_counts()
 
# %%
orders['express'] = orders['express'].str.lower().map({
    'yes': True, 'y':True,'no':False,'n': False
})
 
print(orders['express'].isna().sum())
orders['express'].value_counts()
 
# %%
orders['unit_price'].head(10).tolist()
 
# %%
# orders['unit_price'].astype(float) # Error converting str to float
# price = orders['unit_price'].str.strip()
# print(price.head().tolist())
 
# price = price.str.replace(' kr','')
# print(price.head().tolist())
 
# price = price.str.replace(' ','')
# print(price.head().tolist())
 
# price = price.str.replace(',','.')
# print(price.head().tolist())
 
price = orders['unit_price'].str.replace(r'\s|kr','',regex=True).str.replace(',','')
 
 
 
# %% [markdown]
# normalize price:
 
# %%
orders['unit_price'] = price.astype(float)
 
print(orders['unit_price'].dtype)
orders['unit_price'].describe().round(2)
 
# %% [markdown]
# Dates:pd.to_datetime()
 
# %%
pd.to_datetime(orders['order_date'])
 
# %%
dates = pd.to_datetime(orders['order_date'], errors='coerce') # becomes NaT
orders['order_date'] = dates
print(orders['order_date'].isna().sum()) # missing in data before conversion
 
print(dates.isna().sum()) # missing after
 
orders.loc[dates.isna(),['order_id','order_date']]
 
# %%
#pd.to_datetime(orders['order_date'])
print(orders['order_date'].dtype)
print(orders['order_date'].min())
print(orders['order_date'].max())
print(orders['order_date'].max() - orders['order_date'].min())
 
 
# %% [markdown]
# Orders per period: todo missed in code along
 
# %% [markdown]
# Duplicate: duplicated()
 
# %%
print(orders.duplicated().sum())
 
# %%
orders[orders.duplicated(keep=False)].head(6)
print(orders.duplicated().sum())
 
# %% [markdown]
# Previously we cleaned before checking duplication. Yes this is correct way.
# Because prior to data clean unable to look up deplicate efficiently
 
# %%
print(orders_raw.duplicated().sum())
print(orders.duplicated(subset='order_id').sum())
 
# %%
total_with = (orders['unit_price'] * orders['quantity']).sum()
 
orders = orders.drop_duplicates().reset_index(drop=True)
 
total_without_dup = (orders['unit_price'] * orders['quantity']).sum()
 
print(orders.shape)
print(round(total_with,2))
print(round(total_without_dup,2))
print(round(total_with-total_without_dup,2))
 
# %%
orders = orders.drop(columns='gift_message')
orders.shape
 
# %% [markdown]
# Which category brought in the most money?
 
# %%
orders['total'] = orders['unit_price'] * orders['quantity']
orders.groupby('category')['total'].sum().round(0).sort_values(ascending=False)
 
# %% [markdown]
# Groupby missing column without date silently: Think
 
# %%
monthly = orders.groupby(orders['order_date'].dt.to_period('M'))['total'].sum()
 
print(monthly.round(0))
print(round(monthly.sum(),2))
print(round(orders['total'].sum(),2))
 
# %% [markdown]
# Cleaned data to data folder write as csv file
 
# %%
orders_clean = orders
orders_clean.to_csv('data/orders_clean.csv',index=False)